In [1]:
"""
Plain-Transformers inspection of Mixture-of-Thoughts labels
(no trl, uses built-in chat template, prints human-readable input & label strings)
"""

from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
import torch

# 1. model / tokenizer -------------------------------------------------
model_name = "open-r1/Qwen2.5-Math-7B-RoPE-300k"
tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 2. dataset (first two rows for the demo) -----------------------------
raw_ds = load_dataset("open-r1/Mixture-of-Thoughts", "all", split="train[:2]")

# 3. render each conversation with the tokenizer’s chat template ------
def to_ids(example):
    text = tok.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    # NOTE: add_special_tokens=False → avoids double BOS token
    return tok(text, add_special_tokens=False, truncation=True, max_length=32_768)

ds = raw_ds.map(to_ids, remove_columns=raw_ds.column_names)

# 4. collate (pad + labels) -------------------------------------------
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
batch    = collator([ds[0]])      # inspect a single example
ids      = batch["input_ids"][0]
labels   = batch["labels"][0]

# 5. pretty-print ------------------------------------------------------
def readable(ids_tensor):
    """decode a tensor of token ids, skipping pads (-100)"""
    ids_list = [tid.item() for tid in ids_tensor if tid != -100]
    return tok.decode(ids_list, skip_special_tokens=False)

print("INPUT  (truncated to 300 chars):\n",
      readable(ids)[:300], "\n")
print("LABELS (truncated to 300 chars):\n",
      readable(labels)[:300], "\n")

# 6. dataloader demo ---------------------------------------------------
loader = DataLoader(ds, batch_size=2, collate_fn=collator)
first = next(iter(loader))
print("batch shapes:",
      {k: v.shape for k, v in first.items()})

print)


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

INPUT  (truncated to 300 chars):
 <|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
You will be given a competitive programming problem. Please reason step by step about the solution, then provide a complete implementation in C++17.

Your solution must read input fro 

LABELS (truncated to 300 chars):
 <|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
You will be given a competitive programming problem. Please reason step by step about the solution, then provide a complete implementation in C++17.

Your solution must read input fro 

batch shapes: {'input_ids': torch.Size([2, 22184]), 'attention_mask': torch.Size([2, 22184]), 'labels': torch.Size([2, 22184])}


In [ ]:
batch["labels"][0]